# Fine-tuning PaddleOCR Recognition cho Mã Container

Notebook thực hiện toàn bộ quy trình:
1. Cài đặt PaddlePaddle **3.0.0 (cu126)** — phiên bản duy nhất tương thích với Colab (cuDNN 9.x)
2. Giải nén và crop ảnh từ dataset thô trên Google Drive
3. Tải mô hình pre-trained PP-OCRv3 English
4. Fine-tune mô hình nhận dạng mã container
5. Xuất mô hình inference về Google Drive

> **Tại sao 3.0.0 thay vì 2.6.x?**  
> `paddlepaddle-gpu 2.6.x` được biên dịch với cuDNN **8.x**, nhưng Colab hiện dùng cuDNN **9.8**.  
> Sự không tương thích cuDNN ở mức thư viện C++ gây **Segmentation Fault** không thể vá bằng ENV flags.  
> `paddlepaddle-gpu 3.0.0 (cu126)` được biên dịch với cuDNN **9.x** → tương thích hoàn toàn.


## Bước 1: Kết nối Google Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

## Bước 2A: Cài đặt PaddlePaddle GPU 3.0.0 (cuDNN 9.x compatible)

| | paddlepaddle-gpu 2.6.x | **paddlepaddle-gpu 3.0.0** |
|---|---|---|
| Biên dịch với | cuDNN **8.x** | cuDNN **9.x** |
| Colab cuDNN 9.8 | ❌ Segmentation Fault | ✅ Hoạt động |

**Sau khi cell này xong, Colab có thể yêu cầu khởi động lại — nhấn OK, sau đó tiếp tục từ Bước 2B.**

In [ ]:
# Gỡ phiên bản cũ nếu có
!pip uninstall -y paddlepaddle paddlepaddle-gpu 2>/dev/null || true

# Cài paddlepaddle-gpu 3.0.0 (cu126 - tương thích cuDNN 9.x của Colab)
!pip install paddlepaddle-gpu==3.0.0 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
    -q

# Xác nhận cài đặt và GPU hoạt động
import paddle
print(f'PaddlePaddle version : {paddle.__version__}')
print(f'CUDA compiled        : {paddle.is_compiled_with_cuda()}')
paddle.utils.run_check()

## Bước 2B: Clone PaddleOCR và cài đặt thư viện phụ thuộc

In [ ]:
import os

if not os.path.exists('/content/PaddleOCR'):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
else:
    print('PaddleOCR da ton tai.')

%cd /content/PaddleOCR
!pip install -r requirements.txt -q
print('Hoan tat cai dat PaddleOCR!')

## Bước 3: Giải nén và Crop ảnh Container từ Google Drive

Cell này giải nén `ContainerNum_dataset.zip`, đọc nhãn tọa độ `(x1, y1, x2, y2, label)`,
crop vùng mã container, chia 90/10 Train/Val và tạo file nhãn chuẩn PaddleOCR.

In [ ]:
%cd /content/PaddleOCR

import os, zipfile, cv2, re, random

# ─── Cấu hình đường dẫn ──────────────────────────────────────────────────
OUTER_ZIP       = '/content/drive/MyDrive/ContainerNum_dataset.zip'
TEMP_DATASET    = '/content/temp_dataset'
TEMP_IMAGES     = '/content/temp_train_images'
TEMP_LABELS     = '/content/temp_train_labels'
TRAIN_OUT_DIR   = '/content/PaddleOCR/train_data/rec/train'
VAL_OUT_DIR     = '/content/PaddleOCR/train_data/rec/val'
TRAIN_LABEL_TXT = '/content/PaddleOCR/train_data/rec_train_label.txt'
VAL_LABEL_TXT   = '/content/PaddleOCR/train_data/rec_val_label.txt'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(TRAIN_OUT_DIR, exist_ok=True)
os.makedirs(VAL_OUT_DIR,   exist_ok=True)

print('Dang giai nen ZIP chinh...')
with zipfile.ZipFile(OUTER_ZIP, 'r') as z:
    z.extractall(TEMP_DATASET)

print('Dang giai nen anh tho va nhan...')
with zipfile.ZipFile(f'{TEMP_DATASET}/ContainerNum_dataset/train_images.zip', 'r') as z:
    z.extractall(TEMP_IMAGES)
with zipfile.ZipFile(f'{TEMP_DATASET}/ContainerNum_dataset/train_images_label.zip', 'r') as z:
    z.extractall(TEMP_LABELS)

label_dir   = f'{TEMP_LABELS}/images_label'
label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]

random.seed(42)
random.shuffle(label_files)
split_idx   = int(len(label_files) * 0.9)
train_files = label_files[:split_idx]
val_files   = label_files[split_idx:]
print(f'Tong: {len(label_files)} | Train: {len(train_files)} | Val: {len(val_files)}')


def process_and_crop(files, subset, out_dir, out_label_txt):
    entries = []
    count   = 0
    for lf in files:
        base  = os.path.splitext(lf)[0]
        img_p = os.path.join(f'{TEMP_IMAGES}/images', base + '.jpg')
        lbl_p = os.path.join(label_dir, lf)
        if not os.path.exists(img_p):
            continue
        img = cv2.imread(img_p)
        if img is None:
            continue
        h, w = img.shape[:2]
        with open(lbl_p, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        for idx, line in enumerate(lines):
            parts = line.strip().split(',')
            if len(parts) < 5:
                continue
            try:
                x1  = max(0, int(float(parts[0])))
                y1  = max(0, int(float(parts[1])))
                x2  = min(w,  int(float(parts[2])))
                y2  = min(h,  int(float(parts[3])))
                lbl = re.sub(r'[^A-Z0-9]', '', ','.join(parts[4:]).upper())
                if not lbl or x2 <= x1 or y2 <= y1:
                    continue
                crop_name = f'{base}_{idx}.jpg'
                cv2.imwrite(os.path.join(out_dir, crop_name), img[y1:y2, x1:x2])
                entries.append(f'{crop_name}\t{lbl}\n')
                count += 1
            except Exception:
                pass
    with open(out_label_txt, 'w', encoding='utf-8') as f:
        f.writelines(entries)
    print(f'  [{subset}] Da luu {count} anh crop -> {out_label_txt}')


print('\nDang crop anh...')
process_and_crop(train_files, 'train', TRAIN_OUT_DIR, TRAIN_LABEL_TXT)
process_and_crop(val_files,   'val',   VAL_OUT_DIR,   VAL_LABEL_TXT)

!rm -rf {TEMP_DATASET} {TEMP_IMAGES} {TEMP_LABELS}
print('\nHoan tat xu ly du lieu!')

## Bước 4: Tải Trọng số Pre-trained PP-OCRv3 English

In [ ]:
%cd /content/PaddleOCR

import os

PRETRAIN_DIR   = '/content/PaddleOCR/pretrain_models'
PRETRAIN_CHECK = f'{PRETRAIN_DIR}/en_PP-OCRv3_rec_train/best_accuracy.pdparams'

os.makedirs(PRETRAIN_DIR, exist_ok=True)

if not os.path.exists(PRETRAIN_CHECK):
    print('Dang tai trong so pre-trained...')
    !wget -q --show-progress \
        -P {PRETRAIN_DIR} \
        https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar
    !tar -xf {PRETRAIN_DIR}/en_PP-OCRv3_rec_train.tar -C {PRETRAIN_DIR}
    !rm {PRETRAIN_DIR}/en_PP-OCRv3_rec_train.tar
    print('Tai xong!')
else:
    print('Trong so da ton tai, bo qua tai lai.')

!ls {PRETRAIN_DIR}/en_PP-OCRv3_rec_train/

## Bước 5: Cấu hình File YAML Training

In [ ]:
%cd /content/PaddleOCR

import yaml

CONFIG_PATH = '/content/PaddleOCR/configs/rec/PP-OCRv3/PP-OCRv3_mobile_rec.yml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Global
config['Global']['pretrained_model']    = '/content/PaddleOCR/pretrain_models/en_PP-OCRv3_rec_train/best_accuracy'
config['Global']['save_model_dir']      = '/content/PaddleOCR/output/v3_rec_container/'
config['Global']['character_dict_path'] = '/content/PaddleOCR/ppocr/utils/en_dict.txt'
config['Global']['epoch_num']           = 150
config['Global']['print_batch_step']    = 10
config['Global']['eval_batch_step']     = [0, 2000]
config['Global']['use_gpu']             = True
config['Global']['use_space_char']      = True

# Train dataset
config['Train']['dataset']['data_dir']        = '/content/PaddleOCR/train_data/rec/train/'
config['Train']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data/rec_train_label.txt']

# Loai bo RecConAug va RecAug (augmentation nang, khong can thiet)
train_tfs = config['Train']['dataset'].get('transforms', [])
config['Train']['dataset']['transforms'] = [
    t for t in train_tfs if list(t.keys())[0] not in ('RecConAug', 'RecAug')
]

# Train loader
config['Train']['loader']['batch_size_per_card'] = 64
config['Train']['loader']['num_workers']         = 0  # Tat multiprocess tren Colab
config['Train']['loader']['shuffle']             = True

# Eval dataset
config['Eval']['dataset']['data_dir']        = '/content/PaddleOCR/train_data/rec/val/'
config['Eval']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data/rec_val_label.txt']

# Eval loader
config['Eval']['loader']['batch_size_per_card'] = 64
config['Eval']['loader']['num_workers']         = 0

with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, default_flow_style=False, allow_unicode=True)

print('Da ghi cau hinh YAML thanh cong!')
print(f'  pre-trained : {config["Global"]["pretrained_model"]}')
print(f'  train data  : {config["Train"]["dataset"]["data_dir"]}')
print(f'  val data    : {config["Eval"]["dataset"]["data_dir"]}')
print(f'  batch size  : {config["Train"]["loader"]["batch_size_per_card"]}')
print(f'  num_workers : {config["Train"]["loader"]["num_workers"]}')
print(f'  epochs      : {config["Global"]["epoch_num"]}')

## Bước 6: Huấn luyện Fine-tuning

Với T4 15GB và `batch_size=64`, ước tính ~5-10 phút/epoch.
Model tốt nhất sẽ lưu tại `/content/PaddleOCR/output/v3_rec_container/best_accuracy`.

In [ ]:
%cd /content/PaddleOCR

import paddle
print(f'PaddlePaddle: {paddle.__version__} | GPU: {paddle.device.cuda.device_count()} card(s)')

!python tools/train.py \
    -c configs/rec/PP-OCRv3/PP-OCRv3_mobile_rec.yml

## Bước 7: Xuất Mô hình Inference về Google Drive

In [ ]:
%cd /content/PaddleOCR

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/paddle_rec_inference/'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

!python tools/export_model.py \
    -c configs/rec/PP-OCRv3/PP-OCRv3_mobile_rec.yml \
    -o Global.pretrained_model=/content/PaddleOCR/output/v3_rec_container/best_accuracy \
       Global.save_inference_dir={DRIVE_OUTPUT}

print(f'\nXuat mo hinh thanh cong! Tep tai: {DRIVE_OUTPUT}')
!ls -lh {DRIVE_OUTPUT}